# Interactive exploration: uMLIP-SISSO κ_L

This notebook is a thin wrapper around `umlip_kappa` that lets you:
1. Inspect the benchmark dataset,
2. Sanity-check a single material end-to-end with one uMLIP,
3. Compare the discovered descriptor against the Slack baseline,
4. Explore the softening-bias decomposition.

All heavy lifting is done by the production scripts in `scripts/`; this
notebook is for ad-hoc inspection only.

In [ ]:
import json, sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from umlip_kappa.io_utils import load_config
cfg = load_config('../configs/default.yaml')
cfg['paths']

## 1. Dataset overview

In [ ]:
bench = pd.read_csv('../' + cfg['paths']['benchmark_csv'])
print(bench.shape, '\n', bench.head())
fig, ax = plt.subplots(1, 2, figsize=(8, 3))
ax[0].hist(np.log10(bench['kappa_ref']), bins=20)
ax[0].set_xlabel(r'$\log_{10}\kappa_L$ [W/m/K]')
ax[1].hist(bench['n_atoms_primitive'], bins=20)
ax[1].set_xlabel('atoms in primitive cell')
fig.tight_layout()

## 2. Sanity-check one material end-to-end

Pick a small, well-known insulator (e.g. Si). This cell **runs an actual uMLIP** and takes a few minutes on GPU.

In [ ]:
from umlip_kappa.features_T1 import compute_T1_for_material
from umlip_kappa.features_T2 import compute_T2_for_material

row = bench[bench['mp_id'] == 'mp-149'].iloc[0]
sd = json.loads(row['structure_dict']) if isinstance(row['structure_dict'], str) else row['structure_dict']

t1 = compute_T1_for_material('mp-149', sd, 'mace-mp-0', cfg, force=False)
t2 = compute_T2_for_material('mp-149', sd, 'mace-mp-0', cfg, force=False)
print('Θ_D =', t1['theta_D_K'], 'K')
print('γ_G(300 K) =', t2['gamma_G_300K'])
print('σ^A(300 K) =', t2['sigma_A_300K'])

## 3. Pareto front and best formula

In [ ]:
with open('../outputs/sr/pareto_T2_mace-mp-0.json') as f:
    pf = json.load(f)
pf = sorted(pf, key=lambda r: r['complexity'])
for r in pf:
    print(f"c={r['complexity']:2d}  cv-MAE={r['cv_mae']:.3f}  {r['formula']}")

## 4. Softening-bias decomposition

In [ ]:
decomp = pd.read_csv('../outputs/sr/softening_T2_mace-mp-0_c2.csv')
decomp